> **Historical notebook.** This notebook is preserved for provenance. Cell outputs have been cleared, and the competition data is not included because Zindi's rules prohibit redistribution. See the repository README for data setup; the maintained entry point is `train.py` at the repository root.

# ZINDI Compitition : Basic Needs Basic Rights Kenya - Tech4MentalHealth

### Notebook by : Yassine Daouji
#### Supported by : Azeddine Elhassouny

## Table of contents

1. [Project Discription](#Project-Discription)

2. [Evaluation](#Evaluation)

3. [Data Discription](#Data-Discription)

4. [Required Libraries](#Required-Libraries)

5. [Step 1: Import Data](#Step-1:-Import-Data)

    - [Data Exploration](#Step-1:-Import-Data)

6. [Step 2: Data Preprocessing](#Step-2:-Data-Preprocessing)

    - [LabelEncoder](#Step-2:-Data-Preprocessing)

7. [Step 3: Text Preprocessing](#Step-3:-Text-Preprocessing)

    - [re Librery](#Step-3:-Text-Preprocessing)
 
    - [Stemming with PorterStemme](#Step-3:-Text-Preprocessing)
    
    - [Removing Stopwords using nltk corpus](#Step-3:-Text-Preprocessing)

8. [Step 4: Text Vectorizer](#Step-4:-Text-Vectorizer)

    - [CountVectorizer](#Step-4:-Text-Vectorizer)
 
    - [TfidfVectorizer](#Step-4:-Text-Vectorizer)

9. [Step 5: Features Selection](#Step-5:-Features-Selection)

    - [SelectKBest](#Step-5:-Features-Selection)

    - [chi2](#Step-5:-Features-Selection)

10. [Step 6: train-test Split](#Step-6:-train-test-Split)

11. [Step 7: Classification](#Step-7:-Classification)

    - [LogisticRegression](#Step-7:-Classification)

    - [RandomForestClassifier](#Step-7:-Classification)

12. [Step 8: Models Scoring](#Step-8:-Models-Scoring)

    - [Confusion Matrix](#Step-8:-Models-Scoring)

    - [Accuracy Score](#Step-8:-Models-Scoring)
    
    - [Classification Report](#Step-8:-Models-Scoring)

13. [Predict The classes of the Test Data](#Predict-The-classes-of-the-Test-Data)

14. [Submission](#Submission)

## Project Discription
[[ go back to the top ]](#Table-of-contents)

Around 1 in 4 people will experience a mental health problem this year. Low-income countries have an estimated treatment gap of 85% (as compared with high-income countries with a gap of 35% to 50%). While Kenya has a mental illness prevalence rate that is comparable to that of high-income countries, there are still less than 500 healthcare professionals serving the country.

In Kenya, there are growing concerns about mental health among young people, particularly university students that face a challenging and unique conflation of stressors that put them at risk of challenges like depression and substance abuse.

From the use of app-based solutions for screening to electronically delivered therapies, the use of technologies including machine learning and AI will potentially transform the delivery of mental health services in the coming years.

The objective of this challenge is to develop a machine learning model that classifies statements and questions expressed by university students in Kenya when speaking about the mental health challenges they struggle with. The four categories are depression, suicide, alchoholism, and drug abuse.

This solution will be used for a prototype of a mental health chatbot designed specifically for university students. This initiative is a first step in leveraging technology to make mental health services more accessible and more user-friendly for young people in Kenya and around the world.

#### Basic Needs Basic Rights Kenya (Facebook page):


This challenge is sponsored by Basic Needs Basic Rights (BNBR) Kenya. BNBR supports people with or at increased risk of mental health problems to live and work successfully in their communities by facilitating access to mental health care and social support services.

## Evaluation
[[ go back to the top ]](#Table-of-contents)

The evaluation metric for this challenge is Log Loss.

The values can be between 0 and 1, inclusive. They do not have to add up to 1 across the classes.

## Data Discription
[[ go back to the top ]](#Table-of-contents)

The data consists of statements and questions expressed by students from multiple universities across Kenya who reported suffering from these different mental health challenges.

The wording of the statements is intended to respond to the prompting question, “What is on your mind?”

Note that the written statements may include some spelling or grammatical errors, as well as slang. Also note that labels were not assigned to the statements, but rather students provided statements and questions about their particular experience.

The labels for the training set are contained in Train.csv, corresponding to one of the four categories of mental health problems (depression, suicide, alchoholism, and drug abuse). Your task is to develop a machine learning model to predict the labels for the test set, following the format in sample_submission.csv.

#### Files available for download

* **Train.csv** - has the unique ID of each statement along with the label. You will use this file to train your model on.
* **Test.csv** - has the unique IDs you will be testing your model on.
* **SampleSubmission.csv** - is an example of what your submission file should look like. The order of the rows does not matter, but the order of the columns (Depression, Alcohol, Suicide, Drugs) must be the same as the sample submission. Your submission should contain probabilities that the ID is of each category (with values between 0 and 1 inclusive). The values across the columns do not have to add up to 1.

## Required Libraries
[[ go back to the top ]](#Table-of-contents)

* **NumPy**: Provides a fast numerical array structure and helper functions.
* **pandas**: Provides a DataFrame structure to store data in memory and work with it easily and efficiently.
* **matplotlib**: Basic plotting library in Python; most other Python plotting libraries are built on top of it.
* **scikit-learn**: The essential Machine Learning package in Python.
* **nltk**: The Natural Language Toolkit is a suite of libraries and programs for symbolic and statistical natural language processing for English written in the Python programming language.
* **re**: A regular expression (shortened as regex or regexp, also referred to as rational expression) is a sequence of characters that define a search pattern. Usually such patterns are used by string searching algorithms for "find" or "find and replace" operations on strings, or for input validation. It is a technique developed in theoretical computer science and formal language theory.

## Step 1: Import Data
[[ go back to the top ]](#Table-of-contents)

In [ ]:
import pandas as pd
train_df = pd.read_csv('../data/Train.csv')
test = pd.read_csv('../data/Test.csv')

## Data Exploration
[[ go back to the top ]](#Table-of-contents)

In [ ]:
print(train_df.info())
train_df.head()

In [ ]:
print(test.info())
test.head()

In [ ]:
train_df.label.value_counts()

In [ ]:
import matplotlib.pyplot as plt
train_df.label.value_counts().plot(kind='bar')

## Step 2: Data Preprocessing
[[ go back to the top ]](#Table-of-contents)

In [ ]:
train = train_df.iloc[:, :-1]
target = train_df.iloc[:, -1]

## LabelEncoder
[[ go back to the top ]](#Table-of-contents)

In [ ]:
from sklearn.preprocessing import LabelEncoder
# ALcohol = 0 --- Depression = 1 --- Drugs = 2 --- Suicide = 3 
labelencoder = LabelEncoder()
target = labelencoder.fit_transform(target)
target

## Step 3: Text Preprocessing
[[ go back to the top ]](#Table-of-contents)

In [ ]:
#Concat the Train and the test Data :
text = pd.concat([train, test], ignore_index=True)
text.info

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
corpus = []
for i in range(0, 925):
    # Removing all characters except the letters 
    feeling = re.sub('[^a-zA-Z]', ' ', str(text['text'][i]))
    # Change all teh text to lower case
    feeling = feeling.lower()
    # Spliting text to words to apply stemming and removing stopwords
    feeling = feeling.split()
    ps = PorterStemmer()
    feeling = [ps.stem(word) for word in feeling if not word in set(stopwords.words('english'))]
    # Join the words to get the new text
    feeling = ' '.join(feeling)
    # Add the new text to the corpus list
    corpus.append(feeling)

## Step 4: Text Vectorizer
[[ go back to the top ]](#Table-of-contents)

## CountVectorizer
[[ go back to the top ]](#Table-of-contents)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
Cvectorizer = CountVectorizer()
Cvector_of_words = Cvectorizer.fit_transform(corpus).toarray()
Cvector_of_words

## TfidfVectorizer
[[ go back to the top ]](#Table-of-contents)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
Tvectorizer = TfidfVectorizer()
Tvector_of_words = Tvectorizer.fit_transform(corpus).toarray()
Tvector_of_words

In [ ]:
# Split the vectors of words to test and train (CountVector)
Ctrain_matrix = Cvector_of_words[:616,:]
Ctest_matrix = Cvector_of_words[616:,:]
# Split the vectors of words to test and train (TfidfVector)
Ttrain_matrix = Tvector_of_words[:616,:]
Ttest_matrix = Tvector_of_words[616:,:]

## Step 5: Features Selection
[[ go back to the top ]](#Table-of-contents)

In [ ]:
from sklearn.feature_selection import SelectKBest, chi2
# Selector for CountVec
Cselector = SelectKBest(chi2, k=150)
Ctrain_matrix = Cselector.fit_transform(Ctrain_matrix, target)
# Selector for TfidfVec
Tselector = SelectKBest(chi2, k=50)
Ttrain_matrix = Tselector.fit_transform(Ttrain_matrix, target)

## Step 6: train-test Split
[[ go back to the top ]](#Table-of-contents)

In [ ]:
from sklearn.model_selection import train_test_split
# Train-test-split for CountVec
CX_train, CX_test, Cy_train, Cy_test = train_test_split(Ctrain_matrix, target, test_size = 0.20, random_state = 0)
# Train-test-split for TfidfVec
TX_train, TX_test, Ty_train, Ty_test = train_test_split(Ttrain_matrix, target, test_size = 0.20, random_state = 0)

## Step 7: Classification
[[ go back to the top ]](#Table-of-contents)

## LogisticRegression
[[ go back to the top ]](#Table-of-contents)

In [ ]:
from sklearn.linear_model import LogisticRegression
# CountVec Classifier
CclassifierLR = LogisticRegression(random_state=0)
CclassifierLR.fit(CX_train, Cy_train)
# TfidfVec Classifier
TclassifierLR = LogisticRegression(random_state=0)
TclassifierLR.fit(TX_train, Ty_train)

## RandomForestClassifier
[[ go back to the top ]](#Table-of-contents)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
# CountVec Classifier
CclassifierRF = RandomForestClassifier(n_estimators=100, random_state=0)
CclassifierRF.fit(CX_train, Cy_train)
# TfidfVec Classifier
TclassifierRF = RandomForestClassifier(n_estimators=100, random_state=0)
TclassifierRF.fit(TX_train, Ty_train)

## Step 8: Models Scoring
[[ go back to the top ]](#Table-of-contents)

In [ ]:
#Predict with LogisticRegression to the CountVec
Cy_predLR = CclassifierLR.predict(CX_test)
#Predict with LogisticRegression to the TfidfVec
Ty_predLR = TclassifierLR.predict(TX_test)
#Predict with RandomForest to the CountVec
Cy_predRF = CclassifierRF.predict(CX_test)
#Predict with RandomForest to the TfidfVec
Ty_predRF = TclassifierRF.predict(TX_test)

## Confusion Matrix
[[ go back to the top ]](#Table-of-contents)

In [ ]:
from sklearn.metrics import confusion_matrix
print("The Confusion Matrix of LogisticRegression Classifier with the CountVectorizer : \n",confusion_matrix(Cy_test, Cy_predLR))
print("The Confusion Matrix of LogisticRegression Classifier with the TfidfVectorizer : \n",confusion_matrix(Ty_test, Ty_predLR))
print("The Confusion Matrix of RandomForest Classifier with the CountVectorizer : \n",confusion_matrix(Cy_test, Cy_predRF))
print("The Confusion Matrix of RandomForest Classifier with the TfidfVectorizer : \n",confusion_matrix(Ty_test, Ty_predRF))

## Accuracy Score
[[ go back to the top ]](#Table-of-contents)

In [ ]:
from sklearn.metrics import accuracy_score
print("The Accuracy of LogisticRegression Classifier with the CountVectorizer : ",accuracy_score(Cy_test, Cy_predLR)*100,'%')
print("The Accuracy of LogisticRegression Classifier with the TfidfVectorizer : ",accuracy_score(Ty_test, Ty_predLR)*100,'%')
print("The Accuracy of RandomForest Classifier with the CountVectorizer : ",accuracy_score(Cy_test, Cy_predRF)*100,'%')
print("The Accuracy of RandomForest Classifier with the TfidfVectorizer : ",accuracy_score(Ty_test, Ty_predRF)*100,'%')

## Classification Report
[[ go back to the top ]](#Table-of-contents)

In [ ]:
from sklearn.metrics import classification_report
# ALcohol = 0 --- Depression = 1 --- Drugs = 2 --- Suicide = 3 
print("The Classification Report of LogisticRegression Classifier with the CountVectorizer : \n",
      classification_report(Cy_test, Cy_predLR, target_names=['Alcohol','Depression','Drugs','Suicide']))
print("The Classification Report of LogisticRegression Classifier with the TfidfVectorizer : \n",
      classification_report(Ty_test, Ty_predLR, target_names=['Alcohol','Depression','Drugs','Suicide']))
print("The Classification Report of RandomForest Classifier with the CountVectorizer : \n",
      classification_report(Cy_test, Cy_predRF, target_names=['Alcohol','Depression','Drugs','Suicide']))
print("The Classification Report of RandomForest Classifier with the TfidfVectorizer : \n",
      classification_report(Ty_test, Ty_predRF, target_names=['Alcohol','Depression','Drugs','Suicide']))

## Submission
[[ go back to the top ]](#Table-of-contents)

In [ ]:
# Appling Selector to the test matrix
test_set = Cselector.transform(Ctest_matrix)

# I will submit with the best Vecorizer and the best model : LogisticRegression with the CountVectorizer
y_pred_proba_test_set = CclassifierLR.predict_proba(test_set)

# Create a datarame of submission
Submission = test.iloc[:, 0].to_frame()

# Add the columns of the classes probabilities
# ALcohol = 0 --- Depression = 1 --- Drugs = 2 --- Suicide = 3 
Submission['Depression']=y_pred_proba_test_set[:,1]
Submission['Alcohol']=y_pred_proba_test_set[:,0]
Submission['Suicide']=y_pred_proba_test_set[:,3]
Submission['Drugs']=y_pred_proba_test_set[:,2]

Submission.to_csv('../submissions/Submission.csv', index=False)